# RA-код и belief propagation: короткое введение

Этот ноутбук связывает формулы с работающим Rust декодером. Мы начнём с кода на четыре бита, посмотрим на проверочную матрицу, выполним систематическое кодирование и разберём решение декодера.

**Главная идея:** из `k` информационных битов создаётся слово длины `n`; добавочные биты обеспечивают проверки чётности. Приёмник использует эти проверки и вероятностные сведения о канале, чтобы исправить ошибки.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "Cargo.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "Cargo.toml").exists():
    raise RuntimeError("Open this notebook from inside the ra-rust project")
sys.path.insert(0, str(ROOT / "python"))
from ra_reconcile import decode, decode_batch, encode
MATRIX_SMALL = ROOT / "data" / "H_small.edges"
MATRIX_RA = ROOT / "data" / "RA_n6000_R150.edges"


## Термины и формулы

`H` — бинарная проверочная матрица, `c` — кодовое слово. Допустимое слово удовлетворяет `H cᵀ = 0 (mod 2)`. Результат `H cᵀ` называется синдромом: нулевой синдром означает, что все проверки прошли.

В систематическом RA коде `H=[A|T]`, где `T` — нижняя двухдиагональная матрица. Если `u` — информационные биты, то `v₀ = XOR(A₀·u)`, а `vᵢ = vᵢ₋₁ XOR XOR(Aᵢ·u)`. Кодовое слово `c=(u,v)`, длина сообщения `k=n−m`, скорость `R=k/n`.

Для BSC с ошибкой `p` канал задаёт LLR `Lⱼ=(1−2zⱼ) ln((1−p)/p)`, где `z=A XOR M`. Положительный LLR склоняет бит к 0, отрицательный — к 1. BP передаёт эти мягкие оценки между узлами битов и проверок, пока синдром не станет нулевым или не исчерпается лимит итераций.


In [ ]:
# Матрица имеет два узла проверок и четыре битовых узла.
print(MATRIX_SMALL.read_text())
k, n = 2, 4
print(f"k={k} information bits, n={n} code bits, rate R=k/n={k/n:.2f}")

## Кодирование вручную и через CLI

В маленьком графе проверки имеют вид `c₀ XOR c₁ XOR c₂ = 0` и `c₁ XOR c₂ XOR c₃ = 0`. Для сообщения `10` накопитель даёт проверочные биты `11`, поэтому кодовое слово `1011`.


In [ ]:
word = encode(MATRIX_SMALL, "10")
print("encoded codeword:", word)
assert word == "1011"

## Один пример декодирования

В фиксированном тестовом примере Алиса видит `A=0110`, публичное сообщение `M=1000`, а оценка ошибки канала `p=0.1`. Комбинация `A XOR M=1110` отличается одним битом от допустимого слова `0110`. BP использует уравнения проверок, чтобы восстановить слово.


In [ ]:
result = decode(MATRIX_SMALL, alice="0110", message="1000", p=0.1,
                max_iter=10, threads=1)
print(result)
print("estimate:", result.estimate, "syndrome passed:", result.success)
assert result.estimate == "0110" and result.success

## Что важно помнить

Нулевой синдром не доказывает совпадение с отправленным словом: декодер может сойтись к другому допустимому кодовому слову. Публичные сообщения также раскрывают часть информации о ключе. Полный протокол QKD должен отдельно оценить утечку и выполнить усиление приватности.

Следующие ноутбуки исследуют влияние `p` и статистическую погрешность Монте-Карло.
